# فصل ۱۲: آشنایی با پایگاه داده‌های برداری (Vector Databases)
## بخش ۵: چرخه حیات داده‌ها (CRUD: Create, Read, Update, Delete)

اگر در یک پایگاه داده برداری بدون مدیریت شناسه اقدام به افزودن اسناد کنید، با هر بار اجرای اسکریپت، نسخه‌های تکراری از همان اسناد ایجاد می‌شود! این موضوع باعث هدررفت حافظه، افزایش هزینه‌های پردازشی و مسمومیت نتایج جستجو (Pollution) می‌گردد.

در این بخش، چرخه حیات کامل داده‌ها را در **ChromaDB** و **LangChain** بررسی می‌کنیم:
1. **ایجاد و به‌روزرسانی (Create & Upsert):** مدیریت شناسه‌های یکتا (IDs) و هش‌کردن خودکار با `hashlib`.
2. **خواندن مستقیم (Read / Inspect):** خواندن رکوردها با متد `get` بدون درگیر کردن موتور جستجوی برداری.
3. **حذف (Delete):** پاکسازی اسناد بر اساس شناسه و درک تفاوت Soft Delete و Hard Delete.
4. **تحلیل عمیق امتیازات (Understanding Similarity Scores):** درک فاصله L2، تبدیل به شباهت کسینوسی و تعیین آستانه ارتباط (Threshold).

### ۱. آماده‌سازی محیط و مدل تعبیه متن (Embedding Function)

برای بردارسازی از مدل چندزبانه و محلی **HuggingFace** (`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`) به صورت آفلاین استفاده می‌کنیم.

In [ ]:
import os
from dotenv import load_dotenv

# حذف پروکسی‌های غیرفعال
for var in ["http_proxy", "https_proxy", "HTTP_PROXY", "HTTPS_PROXY"]:
    os.environ.pop(var, None)

os.environ["HF_HUB_OFFLINE"] = "1"
load_dotenv()

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# راه‌اندازی مدل تعبیه متن
embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# مسیر دیتابیس پایدار روی دیسک
persist_directory = "./chroma_db"

# مقداردهی اولیه کالکشن مربوط به این درس
vector_db = Chroma(
    collection_name="my_course_collection",
    embedding_function=embedding_function,
    persist_directory=persist_directory
)

print("✅ دیتابیس Chroma و مدل تعبیه متن با موفقیت آماده شدند.")

### ۲. عملیات ایجاد و آپدیت (Create & Upsert)

اگر هنگام اضافه کردن متون شناسه (ID) ندهید، ChromaDB رشته‌های تصادفی UUID تولید می‌کند. بنابراین با ۱۰ بار اجرا، ۱۰ کپی از هر سند خواهید داشت!

اما اگر **شناسه‌های ثابت و معنادار** تعیین کنید، اکثر وکتور دیتابیس‌ها (از جمله Chroma) عملیات **Upsert** انجام می‌دهند:
> **«اگر این ID وجود نداشت بساز؛ اگر وجود داشت اطلاعات آن را به‌روزرسانی کن.»**

In [ ]:
# داده‌های نمونه
texts = [
    "پایتون یک زبان برنامه‌نویسی عالی برای هوش مصنوعی است.",
    "جاوا اسکریپت زبان اصلی وب است."
]
metadatas = [{"tag": "ai"}, {"tag": "web"}]

# نکته کلیدی: تعریف دستی شناسه‌ها (IDs)
ids = ["lang_python", "lang_js"]

print(f"تعداد اسناد قبل از افزودن: {vector_db._collection.count()}")

# افزودن داده‌ها به دیتابیس (با شناسه‌های مشخص)
vector_db.add_texts(
    texts=texts,
    metadatas=metadatas,
    ids=ids
)

print(f"تعداد اسناد بعد از افزودن: {vector_db._collection.count()}")

#### آزمایش رفتار Upsert:
در سلول زیر همان متون را با همان شناسه‌ها دوباره اضافه می‌کنیم. مشاهده خواهید کرد که تعداد کل اسناد تغییر نمی‌کند، چون رکوردها صرفاً بازنویسی (Overwrite) می‌شوند:

In [ ]:
print(f"تعداد قبل از اجرای مجدد: {vector_db._collection.count()}")

# اجرای مجدد add_texts با همان IDها
vector_db.add_texts(
    texts=texts,
    metadatas=metadatas,
    ids=ids
)

print(f"تعداد بعد از اجرای مجدد: {vector_db._collection.count()} (ثابت ماند!)")

### ۳. راهکار مقیاس‌پذیر: تولید شناسه با هش محتوا (Content Hashing)

در پروژه‌های بزرگ با هزاران سند، نمی‌توان برای هر تکه متن شناسه دستی نوشت. بهترین راهکار استفاده از تابع هش استاندارد مانند **MD5** است.
به این ترتیب هر متن به یک شناسه یکتا تبدیل می‌شود و متن‌های تکراری دقیقاً هش یکسانی تولید کرده و هرگز در دیتابیس تکثیر نمی‌شوند:

In [ ]:
import hashlib

def generate_content_id(text: str) -> str:
    """تولید شناسه ۳۲ کاراکتری بر پایه هش MD5 متن"""
    return hashlib.md5(text.strip().encode("utf-8")).hexdigest()

sample_doc = "مدل‌های زبانی بزرگ از ترنسفورمرها استفاده می‌کنند."
auto_id = generate_content_id(sample_doc)

print(f"متن: {sample_doc}")
print(f"شناسه تولیدشده بر اساس محتوا: {auto_id}")

# افزودن با شناسه خودکار
vector_db.add_texts(
    texts=[sample_doc],
    metadatas=[{"topic": "nlp"}],
    ids=[auto_id]
)
print(f"تعداد اسناد پس از افزودن سند با هش خودکار: {vector_db._collection.count()}")

### ۴. عملیات خواندن مستقیم (Read / Inspect با متد `get`)

گاهی نمی‌خواهید جستجوی معنایی انجام دهید؛ بلکه مستقیماً می‌خواهید محتوای یک سند را بر اساس شناسه بخوانید یا لیست تمام شناسه‌های ذخیره‌شده را استخراج کنید.

متد **`get`** مستقیماً به **لایه انبار داده (Storage Layer / SQLite)** مراجعه می‌کند و بدون نیاز به موتور برداری HNSW داده‌ها را برمی‌گرداند:

In [ ]:
# ۱. دریافت یک سند خاص با ID
record = vector_db.get(ids=["lang_python"])
print("--- دریافت مستقیم با شناسه (Direct Get) ---")
print(f"شناسه‌ها: {record['ids']}")
print(f"متن سند: {record['documents']}")
print(f"متادیتا:  {record['metadatas']}\n")

# ۲. دریافت لیست همه شناسه‌های ذخیره‌شده در کالکشن
all_docs = vector_db.get()
print("--- تمام شناسه‌های موجود در دیتابیس ---")
print(f"تعداد کل: {len(all_docs['ids'])}")
print(f"لیست IDها: {all_docs['ids']}")

### ۵. عملیات حذف (Delete)

برای حذف اسناد، کافی است شناسه‌های مورد نظر را به متد **`delete`** بدهید.

> **نکته فنی (Hard Delete در برابر Soft Delete):**
> در پایگاه‌های داده برداری، حذف بردار از گراف HNSW هزینه‌بر است زیرا ساختار یال‌ها باید بازسازی شود. بنابراین دیتابیس‌ها ابتدا داده را علامت‌گذاری (Soft Delete) می‌کنند تا بلافاصله از دید کاربر پنهان شود، و در زمان‌های بیکاری یا ادغام ایندکس‌ها، پاکسازی کامل را انجام می‌دهند.

In [ ]:
doc_id_to_delete = "lang_js"
print(f"در حال حذف سند با شناسه '{doc_id_to_delete}'...")
print(f"تعداد قبل از حذف: {vector_db._collection.count()}")

# حذف سند
vector_db.delete(ids=[doc_id_to_delete])

print(f"تعداد بعد از حذف: {vector_db._collection.count()}")

# بررسی اینکه آیا سند حذف شده با get پیدا می‌شود یا خیر
deleted_check = vector_db.get(ids=[doc_id_to_delete])
print(f"آیا رکورد هنوز در دیتابیس وجود دارد؟ {len(deleted_check['ids']) > 0}")

### ۶. جستجوی معنایی همراه با امتیاز (Understanding Similarity Scores)

در پروژه‌های عملی نمی‌توان تنها به برگرداندن نزدیک‌ترین اسناد اعتماد کرد؛ باید بدانیم **میزان اطمینان دیتابیس چقدر است؟**
آیا پاسخ واقعاً مرتبط است یا صرفاً «کمترین بی‌ربط» ممکن است؟

برای این منظور از متد **`similarity_search_with_score`** استفاده می‌کنیم.

در ChromaDB، معیار پیش‌فرض **فاصله مربعی اقلیدسی (Squared $L_2$)** است. رابطه بین این فاصله و شباهت کسینوسی برای بردارهای نرمال‌شده به صورت زیر است:

$$\text{Cosine Similarity} = 1 - \frac{\text{Squared } L_2}{2}$$

| مقدار فاصله (Distance Score) | معنا و تفسیر |
| :--- | :--- |
| **`0.0`** | انطباق کامل و ۱۰۰٪ (دقیقاً همان متن) |
| **کمتر از `0.5`** | شباهت معنایی بسیار زیاد (High Relevance) |
| **بین `0.5` تا `1.0`** | شباهت متوسط یا نسبی |
| **بیشتر از `1.0`** | معمولاً بی‌ربط |
| **بیشتر از `1.5`** | کاملاً پرت و نامرتبط |

In [ ]:
# افزودن چند سند جدید با موضوع نجوم برای آزمایش امتیاز
space_docs = [
    "مریخ سیاره سرخ است و چهارمین سیاره منظومه شمسی می‌باشد.",
    "خورشید یک ستاره است که ۹۹ درصد جرم منظومه شمسی را تشکیل می‌دهد.",
    "ماه تنها قمر طبیعی زمین است و انسان در سال ۱۹۶۹ روی آن پا گذاشت."
]
space_ids = ["mars_doc", "sun_doc", "moon_doc"]
vector_db.add_texts(texts=space_docs, ids=space_ids)

query = "مریخ چندمین سیاره منظومه شمسی است؟"
results_with_score = vector_db.similarity_search_with_score(query, k=3)

print(f"🔍 جستجو برای سوال: '{query}'")
print("=" * 60)

for doc, distance_score in results_with_score:
    # تبدیل فاصله Squared L2 به تخمین شباهت کسینوسی
    cosine_sim = 1.0 - (distance_score / 2.0)
    print(f"* متن: {doc.page_content}")
    print(f"  فاصله (Distance):        {distance_score:.4f}  (کمتر = بهتر)")
    print(f"  شباهت کسینوسی (Similarity): {cosine_sim:.4f}  (بیشتر = بهتر)")
    print("-" * 60)

### 🎯 جمع‌بندی نکات کلیدی چرخه حیات داده (CRUD):

1. **همیشه ID تعیین کنید:** تعیین نکردن شناسه باعث تکثیر بی‌رویه اسناد تکراری با هر بار اجرای اسکریپت می‌شود.
2. **عملیات Upsert:** با داشتن شناسه مشخص، متد `add_texts` در صورت وجود سند قبلی، آن را با مقادیر جدید بازنویسی می‌کند.
3. **تولید خودکار شناسه:** استفاده از `hashlib.md5(text)` بهترین راهکار برای تولید دترمینیستیک شناسه از روی محتوا است.
4. **خواندن مستقیم با `get`:** دسترسی مستقیم به متن، شناسه و متادیتای ذخیره‌شده بدون سربار محاسبات برداری.
5. **حذف با `delete(ids=...)`:** پاکسازی اسناد بر اساس شناسه.
6. **امتیاز شباهت:** در معیارهای فاصله، **عدد کوچکتر به معنی شباهت بیشتر** است؛ می‌توان با تعیین آستانه (Threshold) نتایج پرت را از زنجیره LLM فیلتر کرد.